# ปฏิบัติการที่ 5 การค้นหาเฉพาะที่และการหาค่าเหมาะที่สุด: Hill Climbing, Simulated Annealing และ Genetic Algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NKSR22/Introduction_to_AI_Labs/blob/main/lab05-local-search-optimization.ipynb)

ปฏิบัติการประกอบหนังสือ *ปัญญาประดิษฐ์เบื้องต้น* (นครินทร์ ศรีปัญญา) บทที่ 2 หัวข้อ 2.4 การค้นหาเฉพาะที่และการหาค่าเหมาะที่สุด

## วัตถุประสงค์

เมื่อทำปฏิบัติการนี้จบ ผู้เรียนสามารถ

1. เขียน Hill Climbing ตามรหัสเทียมในหนังสือ แล้วอธิบายได้ว่าทำไมผลขึ้นกับจุดเริ่มต้นและติดค่าสูงสุดเฉพาะที่ได้
2. เขียน Simulated Annealing และอธิบายบทบาทของอุณหภูมิ $T$ กับตารางการลดอุณหภูมิได้
3. เปรียบเทียบอัตราความสำเร็จของ Hill Climbing แบบต่าง ๆ และ Simulated Annealing บนปัญหา 8 ราชินีและฟังก์ชันที่มีหลายยอดได้
4. เขียน Genetic Algorithm แก้ปัญหาการจัดสรรกำลังผลิตอย่างประหยัด (Economic Dispatch) และตรวจผลกับคำตอบเชิงวิเคราะห์ได้
5. ทดลองและอธิบายผลของอัตราการกลายพันธุ์ (Mutation Rate) ต่อการลู่เข้าของ GA ได้

## เนื้อหาในหนังสือที่เกี่ยวข้อง

- 2.4.1 Hill Climbing — ตัวอย่างที่ 2.19 และตารางที่ 2.10
- 2.4.2 Simulated Annealing — ตัวอย่างที่ 2.20
- 2.4.3 Genetic Algorithm — ตัวอย่างที่ 2.21 และตารางที่ 2.12

## ความรู้พื้นฐานที่ควรมี

- Python เบื้องต้น ได้แก่ ฟังก์ชัน ลิสต์ ดิกชันนารี และการวนซ้ำ (ทบทวนได้ในภาคผนวก ก)
- อ่านหัวข้อ 2.4 ของหนังสือมาก่อน โดยเฉพาะตัวอย่างที่ 2.19–2.21 เพราะ notebook นี้จะคำนวณตัวอย่างเหล่านั้นซ้ำด้วยโค้ด

## วิธีใช้ notebook นี้

- กดปุ่ม **Open in Colab** ด้านบนเพื่อเปิดใน Google Colab แล้วรันทีละเซลล์จากบนลงล่างด้วย `Shift + Enter`
- ถ้ารันเซลล์ข้ามลำดับแล้วเกิดข้อผิดพลาด ให้เลือกเมนู **Runtime → Run all** เพื่อรันใหม่ทั้งหมด
- เซลล์ที่มี `# TODO` ในส่วน "แบบฝึกปฏิบัติ" คือส่วนที่ผู้เรียนต้องเติมโค้ดเอง
- ทั้ง notebook ใช้เวลารันรวมประมาณ 1 นาที

| ส่วน | หัวข้อ | ตัวอย่างในหนังสือ |
|:---:|---|---|
| 1 | Hill Climbing: จูนอัตราขยาย $K_p$ | 2.19 |
| 2 | Simulated Annealing: หนีจากค่าสูงสุดเฉพาะที่ | 2.20 |
| 3 | ปัญหา 8 ราชินี: เปรียบเทียบอัตราความสำเร็จ | หัวข้อ 2.4.1 |
| 4 | หาค่าสูงสุดของฟังก์ชันที่มีหลายยอด | ตารางที่ 2.12 |
| 5 | GA หนึ่งรุ่นสำหรับ Economic Dispatch 2 เครื่อง | 2.21 |
| 6 | GA เต็มรูปแบบและ Economic Dispatch 3 เครื่อง | ต่อยอด 2.21 |
| 7 | แบบฝึกปฏิบัติ: ผลของอัตราการกลายพันธุ์ | — |

## เตรียมเครื่องมือ

ปฏิบัติการนี้ใช้เฉพาะไลบรารีที่มีใน Google Colab อยู่แล้ว ได้แก่ `numpy` (คำนวณเชิงตัวเลข) `pandas` (แสดงผลเป็นตาราง) และ `matplotlib` (วาดกราฟ) จึงไม่ต้องติดตั้งเพิ่ม

อัลกอริทึมในปฏิบัติการนี้หลายตัวเป็น **อัลกอริทึมเชิงสุ่ม** จึงกำหนด random seed ไว้ เพื่อให้ผลที่ได้เหมือนเดิมทุกครั้งที่รัน และตรงกับคำอธิบายใน notebook

> **หมายเหตุ** ป้ายกำกับบนกราฟใช้ภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ภาษาไทยติดตั้งไว้ ถ้าใช้ภาษาไทยบนกราฟจะแสดงเป็นสี่เหลี่ยม

In [ ]:
# ไม่ต้องติดตั้งไลบรารีเพิ่ม (ถ้ารันบนเครื่องตนเองแล้วขาดไลบรารี ให้รัน: !pip install -q numpy pandas matplotlib)
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42  # กำหนด seed ให้ผลการสุ่มเหมือนเดิมทุกครั้งที่รัน
plt.rcParams["figure.dpi"] = 100

# สีหลักของหนังสือ
BLUE, ORANGE, GREEN, GRAY = "#1F4E79", "#E07B39", "#3A7D44", "#6B6B6B"

print("numpy", np.__version__, "| pandas", pd.__version__)

## ส่วนที่ 1 Hill Climbing: จูนอัตราขยาย $K_p$ (ตัวอย่างที่ 2.19)

### 1.1 ภูมิทัศน์ของปัญหา

วิศวกรต้องการจูน **อัตราขยายสัดส่วน (Proportional Gain)** $K_p$ ของตัวควบคุมความเร็วมอเตอร์ โดยให้คะแนนสมรรถนะจากผลตอบสนองต่อขั้น (Step Response) ยิ่งคะแนนสูงยิ่งดี ตัวอย่างนี้ย่อให้เหลือมิติเดียว คือระดับ $x = 0, 1, \ldots, 10$ ของ $K_p$ และใช้คะแนนสมมติตามตารางที่ 2.10 ของหนังสือ

| $x$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $f(x)$ | 2 | 4 | 7 | **8** | 6 | 5 | 7 | 10 | **12** | 11 | 9 |

องค์ประกอบของการค้นหาเฉพาะที่ (Local Search) ในปัญหานี้คือ

- **สถานะ** ระดับ $x$ หนึ่งค่า (ทุกสถานะเป็นคำตอบที่สมบูรณ์ แม้อาจยังไม่ดี)
- **ฟังก์ชันวัตถุประสงค์ (Objective Function)** คะแนน $f(x)$ ที่ต้องการให้สูงสุด
- **สถานะข้างเคียง (Neighbor)** $x - 1$ และ $x + 1$ ถ้าอยู่ในช่วง 0–10

In [ ]:
# ตารางที่ 2.10: คะแนนสมรรถนะสมมติของระดับ Kp แต่ละระดับ (x = 0..10)
KP_SCORES = [2, 4, 7, 8, 6, 5, 7, 10, 12, 11, 9]


def kp_score(x):
    """ฟังก์ชันวัตถุประสงค์: คะแนนของระดับ x"""
    return KP_SCORES[x]


def kp_neighbors(x):
    """สถานะข้างเคียงของ x คือ x-1 และ x+1 (เฉพาะที่อยู่ในช่วง 0-10)"""
    return [n for n in (x - 1, x + 1) if 0 <= n < len(KP_SCORES)]


print("ข้างเคียงของ x = 0 :", kp_neighbors(0))
print("ข้างเคียงของ x = 3 :", kp_neighbors(3), "คะแนน", [kp_score(n) for n in kp_neighbors(3)])

### 1.2 เขียน Hill Climbing ตามรหัสเทียม

รหัสเทียม HILL-CLIMBING ในหัวข้อ 2.4.1 มีเพียง 3 บรรทัดในวงวน คือ หาข้างเคียงที่ดีที่สุด ถ้าไม่ดีกว่าปัจจุบันให้หยุด มิฉะนั้นย้ายไป ฟังก์ชันด้านล่างเขียนตามนั้นทุกบรรทัด และบันทึกทุกรอบลงในลิสต์ `trace` เพื่อนำไปแสดงเป็นตารางแบบเดียวกับหนังสือ

ฟังก์ชันนี้รับ `value` และ `neighbors` เป็นพารามิเตอร์ จึงใช้ซ้ำกับปัญหาอื่นได้ (ส่วนที่ 3 และ 4)

In [ ]:
def hill_climbing(start, value, neighbors, trace=None):
    """Steepest-ascent Hill Climbing: ย้ายไปข้างเคียงที่ดีที่สุดจนกว่าจะไม่มีข้างเคียงใดดีกว่า

    value     : ฟังก์ชันวัตถุประสงค์ (ต้องการค่าสูงสุด)
    neighbors : ฟังก์ชันคืนรายการสถานะข้างเคียง
    trace     : ลิสต์สำหรับบันทึกแต่ละรอบ (ไม่บังคับ)
    """
    current = start
    round_no = 0
    while True:
        round_no += 1
        candidates = neighbors(current)
        best = max(candidates, key=value)            # ข้างเคียงที่มีค่าสูงที่สุด (ถ้าเสมอ เลือกตัวแรก)
        is_better = value(best) > value(current)
        if trace is not None:
            trace.append({
                "รอบ": round_no,
                "current (f)": f"{current} ({value(current)})",
                "ข้างเคียง (f)": ", ".join(f"{n} ({value(n)})" for n in candidates),
                "ข้างเคียงที่ดีที่สุด": f"{best} ({value(best)})",
                "ดีกว่า?": "✓" if is_better else "✗",
                "การดำเนินการ": f"ย้ายไป {best}" if is_better else f"หยุด คืน x = {current}",
            })
        if not is_better:
            return current
        current = best


for start in [1, 6]:
    trace = []
    result = hill_climbing(start, kp_score, kp_neighbors, trace)
    print(f"กรณีเริ่มที่ x = {start} → ได้ x = {result}, f = {kp_score(result)}")
    display(pd.DataFrame(trace).set_index("รอบ"))

ตารางทั้งสองตรงกับกรณีที่ 1 และ 2 ของตัวอย่างที่ 2.19 ทุกแถว เมื่อเริ่มที่ $x = 1$ Hill Climbing ไปค้างที่ **ค่าสูงสุดเฉพาะที่ (Local Maximum)** $x = 3$ ($f = 8$) ส่วนเมื่อเริ่มที่ $x = 6$ ไปถึง **ค่าสูงสุดวงกว้าง (Global Maximum)** $x = 8$ ($f = 12$)

### 1.3 ผลจากทุกจุดเริ่มต้น และ Random-restart Hill Climbing

หนังสือสรุปว่าจุดเริ่ม $x = 0$ ถึง $4$ ไปค้างที่ $x = 3$ ส่วน $x = 5$ ถึง $10$ ไปถึง $x = 8$ เราตรวจข้อสรุปนี้ได้ด้วยการรันจากทุกจุดเริ่มต้น

In [ ]:
GLOBAL_BEST_X = KP_SCORES.index(max(KP_SCORES))   # x = 8

rows = []
for start in range(len(KP_SCORES)):
    end = hill_climbing(start, kp_score, kp_neighbors)
    rows.append({"จุดเริ่ม x": start, "จบที่ x": end, "f": kp_score(end),
                 "ถึงยอดสูงสุด?": "✓" if end == GLOBAL_BEST_X else "✗"})
start_table = pd.DataFrame(rows).set_index("จุดเริ่ม x")
display(start_table.T)

n_success = (start_table["จบที่ x"] == GLOBAL_BEST_X).sum()
p_success = n_success / len(KP_SCORES)
print(f"จุดเริ่มที่ไปถึงยอดสูงสุด {n_success} จาก {len(KP_SCORES)} ค่า → p = {p_success:.2f}")
print(f"จำนวนรอบการไต่เขาที่คาดว่าต้องทำ (รวมรอบแรก) = 1/p = {1 / p_success:.2f}")

**Random-restart Hill Climbing** ทำ Hill Climbing ซ้ำจากจุดเริ่มสุ่มจนกว่าจะพบคำตอบที่ดีพอ จำนวนรอบที่ต้องทำเป็นตัวแปรสุ่มแบบเรขาคณิต (Geometric) ซึ่งมีค่าคาดหมาย $1/p$ ลองจำลอง 10,000 ครั้งเพื่อตรวจว่าค่าเฉลี่ยใกล้ $1/p \approx 1.83$ หรือไม่

In [ ]:
def random_restart_runs(rng, value, neighbors, n_states, goal):
    """ทำ Hill Climbing จากจุดเริ่มสุ่มซ้ำจนกว่าจะถึง goal คืนจำนวนรอบที่ใช้"""
    runs = 0
    while True:
        runs += 1
        start = rng.randrange(n_states)
        if hill_climbing(start, value, neighbors) == goal:
            return runs


rng = random.Random(RANDOM_SEED)
run_counts = [random_restart_runs(rng, kp_score, kp_neighbors, len(KP_SCORES), GLOBAL_BEST_X)
              for _ in range(10_000)]
print(f"จำนวนรอบเฉลี่ยจากการจำลอง = {np.mean(run_counts):.3f} (ทฤษฎี 1/p = {1 / p_success:.3f})")
print(f"สูงสุดที่พบ = {max(run_counts)} รอบ")

In [ ]:
# วาดภูมิทัศน์และเส้นทางของ Hill Climbing จาก 2 จุดเริ่มต้น
def plot_kp_landscape(paths):
    xs = range(len(KP_SCORES))
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(xs, KP_SCORES, "-o", color=BLUE, label="score f(x)")
    for (start, path), color in zip(paths.items(), [ORANGE, GREEN]):
        ax.plot(path, [kp_score(x) + 0.4 for x in path], "->", color=color,
                label=f"hill climbing from x = {start}")
    ax.set_xlabel("Kp level x")
    ax.set_ylabel("performance score")
    ax.set_xticks(list(xs))
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
    plt.show()


def hill_climbing_path(start):
    """คืนลำดับสถานะที่ Hill Climbing เดินผ่าน"""
    trace = []
    hill_climbing(start, kp_score, kp_neighbors, trace)
    return [start] + [int(row["การดำเนินการ"].split()[-1]) for row in trace if row["ดีกว่า?"] == "✓"]


plot_kp_landscape({1: hill_climbing_path(1), 6: hill_climbing_path(6)})

**คำถามชวนคิด**

1. ถ้าคะแนนที่ $x = 4$ เปลี่ยนจาก 6 เป็น 9 ผลของการเริ่มที่ $x = 1$ จะเปลี่ยนไปอย่างไร และค่า $p$ ของ Random-restart จะเป็นเท่าใด
2. ถ้าเปลี่ยนสถานะข้างเคียงเป็น $x \pm 1$ และ $x \pm 2$ ภูมิทัศน์ "หุบ" ที่ $x = 4, 5$ ยังกั้น Hill Climbing ได้หรือไม่ การเพิ่มขนาดของข้างเคียงมีข้อเสียอะไร

## ส่วนที่ 2 Simulated Annealing: หนีจากค่าสูงสุดเฉพาะที่ (ตัวอย่างที่ 2.20)

### 2.1 เขียน Simulated Annealing ตามรหัสเทียม

**การจำลองการอบอ่อน (Simulated Annealing: SA)** สุ่มเลือกข้างเคียงหนึ่งตัว ถ้าดีกว่ายอมรับเสมอ ถ้าแย่กว่าหรือเท่าเดิมยอมรับด้วยความน่าจะเป็น $e^{\Delta E / T}$ แล้วลดอุณหภูมิแบบเรขาคณิต $T_{k+1} = \alpha T_k$

เพื่อให้ตรวจกับตารางในหนังสือได้ ฟังก์ชันด้านล่างรับ "วิธีเลือกข้างเคียง" (`pick_neighbor`) และ "วิธีสุ่มเลข $r$" (`draw_r`) เป็นพารามิเตอร์ เมื่อใช้งานจริงส่งตัวสุ่มเข้าไป แต่เมื่อต้องการทำซ้ำตัวอย่างที่ 2.20 ส่งลำดับค่าที่หนังสือกำหนดไว้เข้าไปแทน

In [ ]:
def simulated_annealing(start, value, pick_neighbor, draw_r, t_start, alpha, n_steps, trace=None):
    """Simulated Annealing (หาค่าสูงสุด) ด้วยตารางการลดอุณหภูมิแบบเรขาคณิต T <- alpha * T

    pick_neighbor(current) : คืนสถานะข้างเคียงที่สุ่มเลือก
    draw_r()               : คืนเลขสุ่มในช่วง [0, 1) ใช้เมื่อการย้ายแย่ลงหรือเท่าเดิมเท่านั้น
    """
    current = start
    temperature = t_start
    for step in range(1, n_steps + 1):
        candidate = pick_neighbor(current)
        delta_e = value(candidate) - value(current)
        if delta_e > 0:
            p_accept, r, accept = 1.0, None, True          # ดีขึ้น ยอมรับเสมอ
        else:
            p_accept = math.exp(delta_e / temperature)      # แย่ลง ยอมรับด้วยความน่าจะเป็น e^(dE/T)
            r = draw_r()
            accept = r < p_accept
        if trace is not None:
            trace.append({
                "รอบ": step, "T": f"{temperature:.3f}",
                "current (f)": f"{current} ({value(current)})",
                "next ที่สุ่มได้ (f)": f"{candidate} ({value(candidate)})",
                "ΔE": f"{delta_e:+d}" if isinstance(delta_e, int) else f"{delta_e:+.3f}",
                "P ยอมรับ": f"{p_accept:.3f}",
                "r": "—" if r is None else f"{r:.2f}",
                "ผล": f"ยอมรับ → {candidate}" if accept else f"ปฏิเสธ อยู่ที่ {current}",
            })
        if accept:
            current = candidate
        temperature *= alpha
    return current

### 2.2 ทำซ้ำตัวอย่างที่ 2.20 ด้วยลำดับสุ่มเดียวกับหนังสือ

ตัวอย่างที่ 2.20 เริ่มที่ $x = 3$ (จุดที่ Hill Climbing ติดค้าง) ตั้ง $T_1 = 4$ และ $\alpha = 0.7$ และกำหนดข้างเคียงที่สุ่มได้กับเลขสุ่ม $r$ ไว้ล่วงหน้า เราจึงส่งค่าเหล่านี้เข้าไปตามลำดับ

In [ ]:
# ค่าที่ "สุ่มได้" ตามตัวอย่างที่ 2.20 (ใช้ iter เพื่อดึงค่าทีละตัวตามลำดับ)
BOOK_NEXT_STATES = iter([4, 5, 6, 5, 7, 8, 9])
BOOK_R_VALUES = iter([0.42, 0.55, 0.61, 0.35])   # ใช้เฉพาะรอบที่ ΔE <= 0 (รอบที่ 1, 2, 4, 7)

trace = []
result = simulated_annealing(start=3, value=kp_score,
                             pick_neighbor=lambda x: next(BOOK_NEXT_STATES),
                             draw_r=lambda: next(BOOK_R_VALUES),
                             t_start=4.0, alpha=0.7, n_steps=7, trace=trace)
display(pd.DataFrame(trace).set_index("รอบ"))
print(f"จบที่ x = {result}, f = {kp_score(result)}")

ตารางตรงกับตัวอย่างที่ 2.20 ทุกแถว ทั้งค่า $T$ ค่าความน่าจะเป็น และผลการยอมรับ สังเกตว่ารอบที่ 1–2 SA ยอมเดินลง "หุบ" จาก $f = 8$ ไป $f = 5$ ซึ่ง Hill Climbing ไม่มีวันทำ

### 2.3 รัน SA ด้วยเลขสุ่มจริงหลายครั้ง

ตัวอย่างในหนังสือใช้เลขสุ่มที่ "โชคดี" เพื่อสาธิต ข้อควรระวังในหนังสือบอกว่าถ้าเลขสุ่มต่างไป SA อาจติดอยู่ที่ $x = 3$ ลองรัน SA จาก $x = 3$ ครั้งละ 1,000 รอบด้วยการตั้งค่าต่าง ๆ แล้วนับสัดส่วนที่จบที่ยอดสูงสุด $x = 8$

In [ ]:
def sa_success_rate(t_start, alpha, n_steps, n_trials=1000, seed=RANDOM_SEED):
    """สัดส่วนการรัน SA จาก x = 3 ที่จบที่ยอดสูงสุด x = 8"""
    rng = random.Random(seed)
    successes = 0
    for _ in range(n_trials):
        end = simulated_annealing(3, kp_score,
                                  pick_neighbor=lambda x: rng.choice(kp_neighbors(x)),
                                  draw_r=rng.random,
                                  t_start=t_start, alpha=alpha, n_steps=n_steps)
        successes += end == GLOBAL_BEST_X
    return successes / n_trials


settings = [(4.0, 0.70, 7), (4.0, 0.70, 30), (4.0, 0.90, 60), (4.0, 0.95, 120), (4.0, 0.99, 600)]
rows = []
for t_start, alpha, n_steps in settings:
    final_t = t_start * alpha ** n_steps
    rows.append({"T เริ่มต้น": t_start, "alpha": alpha, "จำนวนรอบ": n_steps,
                 "T สุดท้าย": round(final_t, 4),
                 "สัดส่วนที่จบที่ x = 8": sa_success_rate(t_start, alpha, n_steps)})
display(pd.DataFrame(rows))
print("Hill Climbing จาก x = 3 จบที่ x = 8 ได้ 0% เสมอ")

**สิ่งที่ควรสังเกต**

- การตั้งค่าตามหนังสือ ($\alpha = 0.7$ เพียง 7 รอบ) สำเร็จเพียงไม่กี่เปอร์เซ็นต์ เพราะเย็นลงเร็วมาก ตัวอย่างในหนังสือจึงเป็น "กรณีที่โชคดี" กรณีหนึ่ง
- เมื่อลดอุณหภูมิช้าลง ($\alpha$ ใกล้ 1) และให้จำนวนรอบมากพอให้ $T$ ลดจนต่ำ สัดส่วนความสำเร็จเพิ่มขึ้นชัดเจน ตรงกับทฤษฎีที่ว่าถ้าลดอุณหภูมิช้าพอ โอกาสพบค่าสูงสุดวงกว้างจะเข้าใกล้ 1
- การลดช้าต้องแลกกับจำนวนรอบที่มากขึ้น นี่คือการแลกเปลี่ยนระหว่างคุณภาพกับเวลาที่หนังสือกล่าวถึง

**คำถามชวนคิด**

1. ถ้าเริ่มที่ $T = 4$ แต่ตั้ง $\alpha = 0.999$ และรันเพียง 30 รอบ $T$ สุดท้ายจะยังสูงอยู่ ผลลัพธ์ที่ได้จะเป็นอย่างไร (ลองแก้ `settings` แล้วรันดู)
2. ทำไม SA ที่ $T$ สูงมากตลอดเวลาจึงไม่ต่างจากการเดินสุ่ม (Random Walk)

## ส่วนที่ 3 ปัญหา 8 ราชินี: เปรียบเทียบอัตราความสำเร็จ

### 3.1 นิยามปัญหา

ปัญหา 8 ราชินี (8-Queens) ให้วางราชินี 8 ตัวบนกระดานหมากรุก 8 × 8 ไม่ให้กินกัน (ไม่อยู่แถวเดียวกัน คอลัมน์เดียวกัน หรือแนวทแยงเดียวกัน) เป็นปัญหามาตรฐานที่ Russell และ Norvig (2021) ใช้ทดสอบการค้นหาเฉพาะที่

- **สถานะ** วางราชินีคอลัมน์ละ 1 ตัว แทนด้วยทูเพิล `state[col] = row` จึงมี $8^8 \approx 1.7 \times 10^7$ สถานะ
- **ฟังก์ชันวัตถุประสงค์** $h$ = จำนวน **คู่** ราชินีที่กินกันได้ (ต้องการค่า **ต่ำสุด** คำตอบคือ $h = 0$) ใช้กับฟังก์ชัน `hill_climbing` เดิมได้โดยหาค่าสูงสุดของ $-h$
- **สถานะข้างเคียง** ย้ายราชินี 1 ตัวไปแถวอื่นในคอลัมน์เดิม จึงมีข้างเคียง $8 \times 7 = 56$ สถานะ

In [ ]:
N_QUEENS = 8


def attacking_pairs(state):
    """h(state): จำนวนคู่ราชินีที่กินกันได้ (แถวเดียวกันหรือแนวทแยงเดียวกัน)"""
    h = 0
    for c1 in range(len(state)):
        for c2 in range(c1 + 1, len(state)):
            same_row = state[c1] == state[c2]
            same_diagonal = abs(state[c1] - state[c2]) == c2 - c1
            h += same_row or same_diagonal
    return h


def queen_neighbors(state):
    """ข้างเคียงทั้ง 56 สถานะ: ย้ายราชินีหนึ่งตัวไปแถวอื่นในคอลัมน์เดิม"""
    result = []
    for col in range(len(state)):
        for row in range(len(state)):
            if row != state[col]:
                result.append(state[:col] + (row,) + state[col + 1:])
    return result


def random_queen_state(rng, n=N_QUEENS):
    return tuple(rng.randrange(n) for _ in range(n))


def show_board(state):
    """พิมพ์กระดาน Q = ราชินี"""
    for row in range(len(state)):
        print(" ".join("Q" if state[col] == row else "." for col in range(len(state))))


rng = random.Random(RANDOM_SEED)
example_state = random_queen_state(rng)
print("สถานะสุ่ม:", example_state, "| h =", attacking_pairs(example_state),
      "| จำนวนข้างเคียง =", len(queen_neighbors(example_state)))
show_board(example_state)

### 3.2 Hill Climbing สามแบบและ Simulated Annealing

เราจะเปรียบเทียบ 4 วิธีจากหัวข้อ 2.4.1–2.4.2

1. **Steepest-ascent Hill Climbing** — ใช้ฟังก์ชัน `hill_climbing` เดิม ($\text{value} = -h$)
2. **อนุญาตการเดินด้านข้าง (Sideways Move)** — ยอมย้ายไปข้างเคียงที่ค่าเท่ากันได้ติดต่อกันไม่เกิน 100 ครั้ง และเมื่อมีข้างเคียงที่ดีที่สุดหลายตัว ให้สุ่มเลือก
3. **Random-restart Hill Climbing** — เริ่มใหม่จากสถานะสุ่มจนกว่าจะพบคำตอบ
4. **Simulated Annealing** — ใช้ฟังก์ชัน `simulated_annealing` เดิม สุ่มย้ายราชินีหนึ่งตัว

ฟังก์ชันในเซลล์ถัดไปคืนทั้งสถานะสุดท้ายและจำนวนก้าวที่ใช้ เพื่อเปรียบเทียบความเร็วด้วย

In [ ]:
def queen_value(state):
    return -attacking_pairs(state)          # หาค่าสูงสุดของ -h เท่ากับหาค่าต่ำสุดของ h


def steepest_hc_queens(start):
    """Steepest-ascent Hill Climbing ด้วยฟังก์ชัน hill_climbing เดิม คืน (สถานะสุดท้าย, จำนวนก้าว)"""
    trace = []
    final = hill_climbing(start, queen_value, queen_neighbors, trace)
    return final, len(trace) - 1             # รอบสุดท้ายคือรอบที่หยุด ไม่นับเป็นก้าว


def sideways_hc_queens(start, rng, max_sideways=100):
    """Hill Climbing ที่ยอมเดินด้านข้างได้ติดต่อกันไม่เกิน max_sideways ครั้ง"""
    current, steps, sideways = start, 0, 0
    while attacking_pairs(current) > 0:
        scored = [(attacking_pairs(n), n) for n in queen_neighbors(current)]
        best_h = min(h for h, _ in scored)
        current_h = attacking_pairs(current)
        if best_h > current_h or (best_h == current_h and sideways >= max_sideways):
            break                                            # ติดค้าง
        sideways = sideways + 1 if best_h == current_h else 0
        current = rng.choice([n for h, n in scored if h == best_h])   # สุ่มในกลุ่มที่ดีที่สุด
        steps += 1
    return current, steps


def random_restart_queens(rng):
    """เริ่ม Steepest-ascent ใหม่จนกว่าจะได้ h = 0 คืน (สถานะ, จำนวนก้าวรวม, จำนวนรอบ)"""
    total_steps, restarts = 0, 0
    while True:
        restarts += 1
        final, steps = steepest_hc_queens(random_queen_state(rng))
        total_steps += steps
        if attacking_pairs(final) == 0:
            return final, total_steps, restarts


def sa_queens(start, rng, t_start=2.0, alpha=0.995, n_steps=3000):
    """Simulated Annealing สำหรับ 8 ราชินี: ข้างเคียงสุ่ม 1 ตัวจาก 56 ตัว"""
    final = simulated_annealing(start, queen_value,
                                pick_neighbor=lambda s: rng.choice(queen_neighbors(s)),
                                draw_r=rng.random, t_start=t_start, alpha=alpha, n_steps=n_steps)
    return final, n_steps


rng = random.Random(RANDOM_SEED)
final, steps = steepest_hc_queens(example_state)
print(f"Steepest-ascent จากสถานะตัวอย่าง: h = {attacking_pairs(final)} หลัง {steps} ก้าว")
final, steps, restarts = random_restart_queens(rng)
print(f"Random-restart: พบคำตอบ h = 0 หลังเริ่ม {restarts} รอบ รวม {steps} ก้าว")
show_board(final)

### 3.3 ทดลองกับโจทย์สุ่ม

รันแต่ละวิธีกับสถานะเริ่มต้นสุ่มชุดเดียวกัน (Random-restart และ SA ใช้จำนวนครั้งน้อยกว่าเพราะแต่ละครั้งใช้เวลานานกว่า) เซลล์นี้ใช้เวลาประมาณ 20–40 วินาที

In [ ]:
def summarize(name, results, extra=""):
    """สรุปอัตราความสำเร็จและจำนวนก้าวเฉลี่ย results = [(สถานะสุดท้าย, ก้าว), ...]"""
    solved = [steps for final, steps in results if attacking_pairs(final) == 0]
    failed = [steps for final, steps in results if attacking_pairs(final) > 0]
    return {"วิธี": name, "จำนวนโจทย์": len(results),
            "อัตราความสำเร็จ (%)": round(100 * len(solved) / len(results), 1),
            "ก้าวเฉลี่ยเมื่อสำเร็จ": round(np.mean(solved), 1) if solved else None,
            "ก้าวเฉลี่ยเมื่อติดค้าง": round(np.mean(failed), 1) if failed else None,
            "หมายเหตุ": extra}


rng = random.Random(RANDOM_SEED)
starts = [random_queen_state(rng) for _ in range(1000)]

rows = [summarize("Steepest-ascent", [steepest_hc_queens(s) for s in starts]),
        summarize("Sideways (≤ 100)", [sideways_hc_queens(s, rng) for s in starts])]

restart_results = [random_restart_queens(rng) for _ in range(200)]
rows.append(summarize("Random-restart", [(f, steps) for f, steps, _ in restart_results],
                      f"เริ่มเฉลี่ย {np.mean([r for _, _, r in restart_results]):.1f} รอบ"))
rows.append(summarize("Simulated Annealing", [sa_queens(s, rng) for s in starts[:200]],
                      "T = 2, alpha = 0.995, 3000 รอบ"))

queens_table = pd.DataFrame(rows).set_index("วิธี")
display(queens_table)

**เทียบกับหนังสือ** หนังสือ (อ้างอิง Russell & Norvig, 2021) ระบุว่า Steepest-ascent แก้ 8 ราชินีได้ประมาณ 14% และใช้เพียงไม่กี่ก้าว ผลการทดลองข้างบนควรใกล้เคียงค่านี้ Russell และ Norvig ยังรายงานว่าเมื่อยอมเดินด้านข้างได้ 100 ครั้ง อัตราความสำเร็จเพิ่มเป็นประมาณ 94% แต่ใช้ก้าวมากขึ้น (ประมาณ 21 ก้าวเมื่อสำเร็จ และ 64 ก้าวเมื่อติดค้าง) ส่วน Random-restart มีโอกาสสำเร็จต่อรอบ $p \approx 0.14$ จึงคาดว่าต้องทำประมาณ $1/p \approx 7$ รอบ

ตัวเลขที่ได้จะไม่ตรงทุกหลักเพราะเป็นการสุ่ม แต่แนวโน้มควรเหมือนกัน

**คำถามชวนคิด**

1. Random-restart สำเร็จ 100% เสมอ ทำไมจึงยังมีผู้ใช้วิธีอื่น (ดูคอลัมน์จำนวนก้าว)
2. SA ในตารางใช้ 3,000 รอบทุกครั้งแม้พบคำตอบแล้ว ควรเพิ่มเงื่อนไขหยุดอย่างไรจึงจะประหยัดเวลาได้

## ส่วนที่ 4 หาค่าสูงสุดของฟังก์ชันที่มีหลายยอด

ปัญหาเชิงวิศวกรรมหลายปัญหามีตัวแปรต่อเนื่อง เช่น ค่า $K_p$ ที่เป็นจำนวนจริง ส่วนนี้ใช้ฟังก์ชันทดสอบคลาสสิกจากหนังสือของ Michalewicz (1996)

$$f(x) = x \sin(10\pi x) + 1, \qquad -1 \le x \le 2$$

ฟังก์ชันนี้มียอดเล็กยอดน้อยประมาณ 30 ยอด ค่าสูงสุดวงกว้างอยู่ที่ $x \approx 1.8505$ ($f \approx 2.8503$) จึงเป็นภูมิทัศน์แบบ "หลายยอด" ที่หนังสือกล่าวว่า Hill Climbing ทำได้ไม่ดี

ข้างเคียงของ Hill Climbing คือ $x \pm 0.001$ ส่วน SA สุ่มข้างเคียงจาก $x + \varepsilon$ เมื่อ $\varepsilon$ มีการแจกแจงปกติ ส่วนเบี่ยงเบนมาตรฐาน 0.1 (ตัดให้อยู่ในช่วง)

In [ ]:
X_MIN, X_MAX = -1.0, 2.0
STEP = 0.001


def wavy(x):
    """ฟังก์ชันทดสอบของ Michalewicz (1996): f(x) = x sin(10 pi x) + 1"""
    return x * math.sin(10 * math.pi * x) + 1


def wavy_neighbors(x):
    return [max(X_MIN, x - STEP), min(X_MAX, x + STEP)]


# หาค่าสูงสุดวงกว้างด้วยการไล่ตารางละเอียด (ทำได้เพราะมีตัวแปรเดียว)
grid = np.linspace(X_MIN, X_MAX, 300_001)
grid_values = grid * np.sin(10 * np.pi * grid) + 1
X_BEST, F_BEST = grid[grid_values.argmax()], grid_values.max()
print(f"ค่าสูงสุดวงกว้าง: x = {X_BEST:.4f}, f = {F_BEST:.4f}")

rng = random.Random(RANDOM_SEED)
SUCCESS_LEVEL = F_BEST - 0.01        # ถือว่าสำเร็จถ้าได้ f ต่างจากค่าสูงสุดไม่เกิน 0.01

hc_finals = [hill_climbing(rng.uniform(X_MIN, X_MAX), wavy, wavy_neighbors) for _ in range(200)]


def sa_wavy(rng, n_steps=2000):
    clip = lambda x: min(X_MAX, max(X_MIN, x))
    return simulated_annealing(rng.uniform(X_MIN, X_MAX), wavy,
                               pick_neighbor=lambda x: clip(x + rng.gauss(0, 0.1)),
                               draw_r=rng.random, t_start=1.0, alpha=0.997, n_steps=n_steps)


sa_finals = [sa_wavy(rng) for _ in range(200)]

# Random-restart: ทำ Hill Climbing 10 รอบต่อ 1 ครั้งทดลอง แล้วเลือกผลที่ดีที่สุด
restart_finals = [max((hill_climbing(rng.uniform(X_MIN, X_MAX), wavy, wavy_neighbors) for _ in range(10)),
                      key=wavy) for _ in range(200)]

rows = []
for name, finals in [("Hill Climbing", hc_finals), ("Random-restart (10 รอบ)", restart_finals),
                     ("Simulated Annealing", sa_finals)]:
    values = [wavy(x) for x in finals]
    rows.append({"วิธี": name, "f เฉลี่ย": round(np.mean(values), 3),
                 "อัตราความสำเร็จ (%)": round(100 * np.mean([v >= SUCCESS_LEVEL for v in values]), 1)})
display(pd.DataFrame(rows).set_index("วิธี"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(grid, grid_values, color=BLUE, lw=1, label="f(x) = x sin(10 pi x) + 1")
ax.plot(hc_finals, [wavy(x) for x in hc_finals], "o", color=GRAY, ms=4, alpha=0.6, label="hill climbing end points")
ax.plot(sa_finals, [wavy(x) + 0.08 for x in sa_finals], "v", color=ORANGE, ms=4, alpha=0.6,
        label="simulated annealing end points")
ax.plot(X_BEST, F_BEST, "*", color=GREEN, ms=15, label="global maximum")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.3)
plt.show()

จุดสีเทากระจายอยู่บนยอดเล็กยอดน้อยทั่วกราฟ เพราะ Hill Climbing หยุดที่ยอดแรกที่ไต่ถึง ส่วน SA ซึ่งยอมเดินลงหุบได้ตอนอุณหภูมิสูง จะไปรวมกันที่ยอดสูง ๆ ทางขวามากกว่า

**คำถามชวนคิด** ถ้าเพิ่ม STEP ของ Hill Climbing จาก 0.001 เป็น 0.05 ผลจะดีขึ้นหรือแย่ลง (ลองแก้แล้วรันดู) ทำไม

## ส่วนที่ 5 GA หนึ่งรุ่นสำหรับ Economic Dispatch 2 เครื่อง (ตัวอย่างที่ 2.21)

### 5.1 โจทย์และการเข้ารหัส

**การจัดสรรกำลังผลิตอย่างประหยัด (Economic Dispatch: ED)** คือการแบ่งความต้องการไฟฟ้าให้เครื่องกำเนิดไฟฟ้าแต่ละเครื่อง โดยต้นทุนเชื้อเพลิงรวมต่ำที่สุด ตัวอย่างที่ 2.21 (ค่าสมมติ) มีความต้องการไฟฟ้า $D = 150$ MW แบ่งให้เครื่อง G1 (50–125 MW) และ G2 (25–100 MW)

$$C_1(P_1) = 0.02P_1^2 + 2P_1, \qquad C_2(P_2) = 0.04P_2^2 + 1.5P_2$$

- **การเข้ารหัส** โครโมโซม 4 บิตแทนจำนวนเต็ม $k = 0$–$15$ แล้วแปลงเป็น $P_1 = 50 + 5k$ MW และ $P_2 = 150 - P_1$ (อยู่ในขีดจำกัดเสมอ)
- **ต้นทุนรวม** $F(P_1) = C_1(P_1) + C_2(150 - P_1) = 0.06P_1^2 - 11.5P_1 + 1125$
- **ความเหมาะสม** $\text{fitness} = 700 - F$ (700 คือต้นทุนสูงสุดในช่วงนี้ ที่ $k = 0$)

In [ ]:
DEMAND_MW = 150


def cost_g1(p1):
    return 0.02 * p1 ** 2 + 2 * p1


def cost_g2(p2):
    return 0.04 * p2 ** 2 + 1.5 * p2


def decode_dispatch(bits):
    """แปลงโครโมโซม 4 บิตเป็น (k, P1, P2) ตามตัวอย่างที่ 2.21"""
    k = int(bits, 2)
    p1 = 50 + 5 * k
    return k, p1, DEMAND_MW - p1


def dispatch_cost(bits):
    """ต้นทุนเชื้อเพลิงรวม F (หน่วยเงินต่อชั่วโมง)"""
    _, p1, p2 = decode_dispatch(bits)
    return cost_g1(p1) + cost_g2(p2)


def dispatch_fitness(bits):
    return 700 - dispatch_cost(bits)


print("ตรวจตัวอย่างในหนังสือ: 1001 →", decode_dispatch("1001"), "F =", dispatch_cost("1001"))

### 5.2 ประชากรเริ่มต้นและการคัดเลือกแบบวงล้อรูเล็ตต์

**การคัดเลือกแบบวงล้อรูเล็ตต์ (Roulette-wheel Selection)** ให้โครโมโซม $i$ มีโอกาสถูกเลือก $p_i = \text{fitness}_i / \sum_j \text{fitness}_j$ แล้วหมุนวงล้อด้วยเลขสุ่ม $r$ ฟังก์ชัน `roulette_pick` รับ $r$ เข้ามาตรง ๆ เพื่อใช้เลขสุ่มชุดเดียวกับหนังสือ (0.31, 0.12, 0.88, 0.65)

In [ ]:
def roulette_pick(population, fitnesses, r):
    """คืนดัชนีของโครโมโซมที่ r (0 <= r < 1) ตกอยู่ในช่วงสะสมของมัน"""
    total = sum(fitnesses)
    cumulative = 0.0
    for index, fit in enumerate(fitnesses):
        cumulative += fit / total
        if r < cumulative:
            return index
    return len(population) - 1


def population_table(population, names):
    """ตารางประชากรแบบเดียวกับขั้นที่ 3 ของตัวอย่างที่ 2.21"""
    fitnesses = [dispatch_fitness(c) for c in population]
    total = sum(fitnesses)
    rows, cumulative = [], 0.0
    for name, bits, fit in zip(names, population, fitnesses):
        k, p1, p2 = decode_dispatch(bits)
        start, cumulative = cumulative, cumulative + fit / total
        rows.append({"โครโมโซม": name, "บิต": bits, "k": k, "P1 / P2 (MW)": f"{p1} / {p2}",
                     "F": round(dispatch_cost(bits), 2), "fitness": round(fit, 2),
                     "p_i": round(fit / total, 3), "ช่วงสะสม": f"{start:.3f}–{cumulative:.3f}"})
    return pd.DataFrame(rows).set_index("โครโมโซม")


GEN0 = ["0011", "1110", "0110", "1101"]
NAMES = ["c1", "c2", "c3", "c4"]
display(population_table(GEN0, NAMES))

gen0_fitness = [dispatch_fitness(c) for c in GEN0]
BOOK_SPINS = [0.31, 0.12, 0.88, 0.65]
picked = [roulette_pick(GEN0, gen0_fitness, r) for r in BOOK_SPINS]
print("หมุนวงล้อได้:", [NAMES[i] for i in picked])

### 5.3 การไขว้เปลี่ยนจุดเดียวและการกลายพันธุ์

**การไขว้เปลี่ยนจุดเดียว (Single-point Crossover)** ตัดโครโมโซมพ่อแม่ที่จุดเดียวกัน แล้วสลับส่วนหลัง ส่วน **การกลายพันธุ์ (Mutation)** แบบกลับบิต (Bit-flip) เปลี่ยนบิต 0 เป็น 1 หรือ 1 เป็น 0 ตามหนังสือ คู่ที่ 1 ตัดหลังบิตที่ 1 คู่ที่ 2 ตัดหลังบิตที่ 3 และบิตที่ 3 ของลูก `1011` ถูกกลับค่า

In [ ]:
def crossover_at(parent1, parent2, point):
    """ไขว้เปลี่ยนจุดเดียว ตัดหลังบิตที่ point (นับจาก 1)"""
    return parent1[:point] + parent2[point:], parent2[:point] + parent1[point:]


def flip_bit(bits, position):
    """กลับค่าบิตที่ position (นับจาก 1 ทางซ้าย)"""
    i = position - 1
    return bits[:i] + ("0" if bits[i] == "1" else "1") + bits[i + 1:]


# คู่ที่ 1 = (c2, c1) ตัดหลังบิตที่ 1, คู่ที่ 2 = (c4, c3) ตัดหลังบิตที่ 3
pairs = [(GEN0[picked[0]], GEN0[picked[1]], 1), (GEN0[picked[2]], GEN0[picked[3]], 3)]
rows, children = [], []
for pair_no, (father, mother, point) in enumerate(pairs, start=1):
    for child in crossover_at(father, mother, point):
        k, p1, _ = decode_dispatch(child)
        rows.append({"คู่": pair_no, "พ่อ × แม่": f"{father[:point]}|{father[point:]} × {mother[:point]}|{mother[point:]}",
                     "ลูก": child, "k": k, "P1": p1, "F": dispatch_cost(child)})
        children.append(child)
display(pd.DataFrame(rows))

# ขั้นที่ 5: บิตที่ 3 ของลูก 1011 ถูกกลับค่า
children[0] = flip_bit(children[0], 3)
print("หลังกลายพันธุ์:", children[0], "→", decode_dispatch(children[0]), "F =", dispatch_cost(children[0]))
GEN1 = children

In [ ]:
# ขั้นที่ 6: ประเมินรุ่นใหม่ เทียบกับรุ่น 0
rows = []
for gen, population in [(0, GEN0), (1, GEN1)]:
    costs = [dispatch_cost(c) for c in population]
    rows.append({"รุ่น": gen, "ประชากร": ", ".join(population),
                 "ต้นทุนดีที่สุด": min(costs), "ต้นทุนเฉลี่ย": np.mean(costs)})
display(pd.DataFrame(rows).set_index("รุ่น"))

### 5.4 ตรวจกับคำตอบเชิงวิเคราะห์ (ต้นทุนส่วนเพิ่มเท่ากัน)

ทฤษฎี ED บอกว่าคำตอบที่ดีที่สุด (เมื่อไม่ติดขีดจำกัด) คือจุดที่ **ต้นทุนส่วนเพิ่ม (Incremental Cost)** $dC_i/dP_i$ ของทุกเครื่องเท่ากัน เรียกค่านี้ว่า $\lambda$

$$0.04P_1 + 2 = 0.08P_2 + 1.5, \qquad P_1 + P_2 = 150$$

เซลล์ถัดไปแก้สมการนี้ แล้วเทียบกับการไล่ทุกค่า $k$ ทั้ง 16 ค่า (ทำได้เพราะปัญหาเล็ก)

In [ ]:
# ต้นทุนส่วนเพิ่ม: dC1/dP1 = 0.04 P1 + 2, dC2/dP2 = 0.08 P2 + 1.5
# แทน P2 = 150 - P1: 0.04 P1 + 2 = 0.08 (150 - P1) + 1.5  →  0.12 P1 = 11.5
p1_opt = 11.5 / 0.12
p2_opt = DEMAND_MW - p1_opt
lam = 0.04 * p1_opt + 2
print(f"คำตอบเชิงวิเคราะห์: P1 = {p1_opt:.2f} MW, P2 = {p2_opt:.2f} MW, "
      f"lambda = {lam:.3f}, F = {cost_g1(p1_opt) + cost_g2(p2_opt):.2f}")
print(f"ตรวจ: dC2/dP2 = {0.08 * p2_opt + 1.5:.3f} (ต้องเท่ากับ lambda)")

all_bits = [format(k, "04b") for k in range(16)]
brute = pd.DataFrame([{"บิต": b, "k": decode_dispatch(b)[0], "P1": decode_dispatch(b)[1],
                       "F": dispatch_cost(b)} for b in all_bits]).set_index("บิต")
best_bits = brute["F"].idxmin()
print(f"ไล่ทุกค่า k: ดีที่สุดคือ {best_bits} (P1 = {brute.loc[best_bits, 'P1']} MW, F = {brute.loc[best_bits, 'F']})")
display(brute.T)

In [ ]:
p1_curve = np.linspace(50, 125, 200)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(p1_curve, 0.06 * p1_curve ** 2 - 11.5 * p1_curve + 1125, color=BLUE, label="F(P1)")
ax.plot([decode_dispatch(c)[1] for c in GEN0], [dispatch_cost(c) for c in GEN0], "o", color=GRAY,
        ms=8, label="generation 0")
ax.plot([decode_dispatch(c)[1] for c in GEN1], [dispatch_cost(c) for c in GEN1], "s", color=ORANGE,
        ms=7, label="generation 1")
ax.axvline(p1_opt, color=GREEN, ls="--", label=f"equal incremental cost P1 = {p1_opt:.1f} MW")
ax.set_xlabel("P1 (MW)")
ax.set_ylabel("total cost F")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

ผลทุกขั้นตรงกับตัวอย่างที่ 2.21 ได้แก่ ต้นทุนเฉลี่ยลดจาก 606.25 เป็น 582.50 และรุ่นที่ 1 มีโครโมโซม `1001` ($P_1 = 95$ MW, $F = 574$) ซึ่งเป็นค่าที่ดีที่สุดของการเข้ารหัสทีละ 5 MW ใกล้กับคำตอบเชิงวิเคราะห์ $P_1 \approx 95.8$ MW มากที่สุด

**คำถามชวนคิด** ถ้าต้องการความละเอียด 1 MW แทน 5 MW ในช่วง $P_1$ = 50–125 MW ต้องใช้โครโมโซมกี่บิต

## ส่วนที่ 6 GA เต็มรูปแบบและ Economic Dispatch 3 เครื่อง

### 6.1 ฟังก์ชัน GA ทั่วไป

ส่วนที่ 5 ทำทีละขั้นด้วยมือ ส่วนนี้รวมทุกขั้นเป็นฟังก์ชันเดียวตามรหัสเทียม GENETIC-ALGORITHM ในหัวข้อ 2.4.3 โดยเพิ่ม

- **การคัดเลือกแบบแข่งขัน (Tournament Selection)** สุ่มหยิบ 2 ตัวแล้วเลือกตัวที่ดีกว่า ไม่ขึ้นกับสเกลของ fitness (ใช้ได้แม้ fitness ติดลบ) เป็นค่าเริ่มต้น และยังเลือกวงล้อรูเล็ตต์ได้
- **การเก็บรักษาตัวที่ดีที่สุด (Elitism)** คัดลอกตัวที่ดีที่สุดไปรุ่นถัดไปโดยตรง
- **อัตราการไขว้เปลี่ยน** $p_c$ และ **อัตราการกลายพันธุ์ต่อบิต** $p_m$

ฟังก์ชันคืนโครโมโซมที่ดีที่สุดและประวัติ fitness ของแต่ละรุ่น (ดีที่สุดและเฉลี่ย) เพื่อนำไปวาดกราฟการลู่เข้า

In [ ]:
def random_chromosome(n_bits, rng):
    return "".join(rng.choice("01") for _ in range(n_bits))


def tournament_select(population, fitnesses, rng, k=2):
    """สุ่มหยิบ k ตัว แล้วคืนตัวที่ fitness สูงที่สุด"""
    contestants = rng.sample(range(len(population)), k)
    return population[max(contestants, key=lambda i: fitnesses[i])]


def roulette_select(population, fitnesses, rng):
    """วงล้อรูเล็ตต์ (fitness ต้องเป็นบวก)"""
    return population[roulette_pick(population, fitnesses, rng.random())]


def single_point_crossover(parent1, parent2, rng, pc):
    """ไขว้เปลี่ยนจุดเดียวด้วยความน่าจะเป็น pc มิฉะนั้นคืนสำเนาของพ่อแม่"""
    if rng.random() >= pc:
        return parent1, parent2
    point = rng.randint(1, len(parent1) - 1)
    return crossover_at(parent1, parent2, point)


def bit_flip_mutation(bits, pm, rng):
    """กลับค่าแต่ละบิตอย่างอิสระด้วยความน่าจะเป็น pm"""
    return "".join(("1" if b == "0" else "0") if rng.random() < pm else b for b in bits)


def genetic_algorithm(fitness_fn, n_bits, pop_size=30, n_generations=50, pc=0.9, pm=0.01,
                      selection="tournament", elitism=1, seed=RANDOM_SEED):
    """GA แบบเข้ารหัสบิต คืน (โครโมโซมที่ดีที่สุด, fitness ที่ดีที่สุด, ประวัติ)"""
    rng = random.Random(seed)
    select = tournament_select if selection == "tournament" else roulette_select
    population = [random_chromosome(n_bits, rng) for _ in range(pop_size)]
    history = {"best": [], "mean": []}
    best_chrom, best_fit = None, -math.inf
    for generation in range(n_generations + 1):
        fitnesses = [fitness_fn(c) for c in population]
        for chrom, fit in zip(population, fitnesses):          # จำตัวที่ดีที่สุดที่เคยพบ
            if fit > best_fit:
                best_chrom, best_fit = chrom, fit
        history["best"].append(max(fitnesses))
        history["mean"].append(float(np.mean(fitnesses)))
        if generation == n_generations:
            break
        ranked = sorted(range(pop_size), key=lambda i: fitnesses[i], reverse=True)
        next_population = [population[i] for i in ranked[:elitism]]   # elitism
        while len(next_population) < pop_size:
            parent1 = select(population, fitnesses, rng)
            parent2 = select(population, fitnesses, rng)
            for child in single_point_crossover(parent1, parent2, rng, pc):
                next_population.append(bit_flip_mutation(child, pm, rng))
        population = next_population[:pop_size]
    return best_chrom, best_fit, history


# ทดสอบกับ ED 2 เครื่องของส่วนที่ 5: วงล้อรูเล็ตต์ ไขว้เปลี่ยนทุกคู่ pm = 0.05 เหมือนหนังสือ
# (เพิ่มประชากรเป็น 6 ตัวและเก็บตัวที่ดีที่สุด 1 ตัว)
best, fit, hist = genetic_algorithm(dispatch_fitness, n_bits=4, pop_size=6, n_generations=10,
                                    pc=1.0, pm=0.05, selection="roulette", elitism=1)
print("ED 2 เครื่อง: ดีที่สุด", best, decode_dispatch(best), "F =", dispatch_cost(best))
print("ต้นทุนดีที่สุดแต่ละรุ่น:", [round(700 - f, 1) for f in hist["best"]])

### 6.2 Economic Dispatch 3 เครื่อง

ต่อยอดไปสู่เครื่องกำเนิด 3 เครื่อง ใช้ข้อมูลจากตัวอย่าง 3A ในหนังสือ *Power Generation, Operation, and Control* ของ Wood et al. (2014) ซึ่งเป็นโจทย์มาตรฐานของวิชาการเดินเครื่องระบบไฟฟ้ากำลัง ความต้องการไฟฟ้า $D = 850$ MW

| เครื่อง | ต้นทุน $C_i(P_i)$ (หน่วยเงินต่อชั่วโมง) | ขีดจำกัด (MW) |
|:---:|---|:---:|
| G1 | $561 + 7.92P_1 + 0.001562P_1^2$ | 150–600 |
| G2 | $310 + 7.85P_2 + 0.00194P_2^2$ | 100–400 |
| G3 | $78 + 7.97P_3 + 0.00482P_3^2$ | 50–200 |

**การเข้ารหัส** $P_1$ และ $P_2$ ใช้ 10 บิตต่อเครื่อง แปลงเป็นค่าในขีดจำกัดของตนแบบเชิงเส้น แล้วให้ $P_3 = D - P_1 - P_2$ เพื่อให้กำลังผลิตรวมเท่ากับความต้องการเสมอ แต่ $P_3$ อาจหลุดขีดจำกัด จึงบวก **ค่าปรับ (Penalty)** เข้าไปในต้นทุนตามระยะที่หลุด ความเหมาะสมคือ $-(\text{ต้นทุน} + \text{ค่าปรับ})$ ซึ่งติดลบ จึงต้องใช้การคัดเลือกแบบแข่งขัน

In [ ]:
# ข้อมูล: Wood, Wollenberg & Sheble (2014) ตัวอย่าง 3A — (c, b, a) ของ C = c + bP + aP^2 และขีดจำกัด
UNITS = {
    "G1": {"c": 561, "b": 7.92, "a": 0.001562, "pmin": 150, "pmax": 600},
    "G2": {"c": 310, "b": 7.85, "a": 0.00194, "pmin": 100, "pmax": 400},
    "G3": {"c": 78, "b": 7.97, "a": 0.00482, "pmin": 50, "pmax": 200},
}
DEMAND_3 = 850
BITS_PER_UNIT = 10
PENALTY_PER_MW = 1000          # ค่าปรับต่อ MW ที่หลุดขีดจำกัด


def unit_cost(name, p):
    u = UNITS[name]
    return u["c"] + u["b"] * p + u["a"] * p ** 2


def decode_power(bits, name):
    """แปลงบิตเป็นกำลังผลิตในช่วง [pmin, pmax] แบบเชิงเส้น"""
    u = UNITS[name]
    return u["pmin"] + int(bits, 2) / (2 ** len(bits) - 1) * (u["pmax"] - u["pmin"])


def decode_ed3(chrom):
    p1 = decode_power(chrom[:BITS_PER_UNIT], "G1")
    p2 = decode_power(chrom[BITS_PER_UNIT:], "G2")
    return p1, p2, DEMAND_3 - p1 - p2


def ed3_cost_with_penalty(chrom):
    p1, p2, p3 = decode_ed3(chrom)
    u3 = UNITS["G3"]
    violation = max(0, u3["pmin"] - p3) + max(0, p3 - u3["pmax"])
    cost = unit_cost("G1", p1) + unit_cost("G2", p2) + unit_cost("G3", p3)
    return cost + PENALTY_PER_MW * violation


def ed3_fitness(chrom):
    return -ed3_cost_with_penalty(chrom)

**คำตอบอ้างอิงด้วย Lambda Iteration** เมื่อไม่ติดขีดจำกัด ทุกเครื่องมีต้นทุนส่วนเพิ่มเท่ากับ $\lambda$ คือ $b_i + 2a_iP_i = \lambda$ จึงได้ $P_i = (\lambda - b_i)/(2a_i)$ วิธี **Lambda Iteration** ค้นหา $\lambda$ ด้วยการแบ่งครึ่งช่วง (Bisection) จนกว่า $\sum P_i = D$ (Wood et al., 2014)

In [ ]:
def lambda_iteration(demand, tol=1e-6):
    """หา lambda ที่ทำให้ผลรวมกำลังผลิตเท่ากับ demand (ตัดค่าให้อยู่ในขีดจำกัดของแต่ละเครื่อง)"""
    def outputs(lam):
        return {n: min(u["pmax"], max(u["pmin"], (lam - u["b"]) / (2 * u["a"]))) for n, u in UNITS.items()}
    low, high = 0.0, 50.0
    while high - low > tol:
        mid = (low + high) / 2
        if sum(outputs(mid).values()) < demand:
            low = mid
        else:
            high = mid
    return (low + high) / 2, outputs((low + high) / 2)


lam3, p_opt3 = lambda_iteration(DEMAND_3)
cost_opt3 = sum(unit_cost(n, p) for n, p in p_opt3.items())
print(f"Lambda Iteration: lambda = {lam3:.3f}  " +
      ", ".join(f"{n} = {p:.1f} MW" for n, p in p_opt3.items()) + f"  ต้นทุนรวม = {cost_opt3:.2f}")

ผลควรได้ $\lambda \approx 9.148$, $P_1 \approx 393.2$, $P_2 \approx 334.6$ และ $P_3 \approx 122.2$ MW ตรงกับคำตอบในหนังสือของ Wood et al. (2014) ต่อไปลองให้ GA หาคำตอบโดยไม่รู้สูตรต้นทุนส่วนเพิ่มเลย รู้เพียงวิธีคำนวณต้นทุนของแต่ละโครโมโซม

In [ ]:
best3, fit3, hist3 = genetic_algorithm(ed3_fitness, n_bits=2 * BITS_PER_UNIT, pop_size=40,
                                       n_generations=80, pc=0.9, pm=0.02)
p1, p2, p3 = decode_ed3(best3)
ga_cost = -fit3
print(f"GA: P1 = {p1:.1f}, P2 = {p2:.1f}, P3 = {p3:.1f} MW  ต้นทุนรวม = {ga_cost:.2f}")
print(f"ต่างจาก Lambda Iteration {ga_cost - cost_opt3:.2f} หน่วยเงินต่อชั่วโมง "
      f"({100 * (ga_cost - cost_opt3) / cost_opt3:.4f}%)")
print(f"จำนวนครั้งที่ประเมินต้นทุน = {40 * 81:,} ครั้ง จากโครโมโซมที่เป็นไปได้ {2 ** 20:,} แบบ")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([-f for f in hist3["best"]], color=BLUE, label="best cost in generation")
ax.plot([-f for f in hist3["mean"]], color=ORANGE, label="mean cost in generation")
ax.axhline(cost_opt3, color=GREEN, ls="--", label="lambda iteration optimum")
ax.set_ylim(cost_opt3 - 20, cost_opt3 + 300)
ax.set_xlabel("generation")
ax.set_ylabel("total cost (incl. penalty)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

GA ได้คำตอบที่ใกล้คำตอบเชิงวิเคราะห์มาก (ต่างกันเพียงเศษส่วนของร้อยละ ซึ่งส่วนหนึ่งมาจากความละเอียดของการเข้ารหัส 10 บิต) ทั้งที่ประเมินโครโมโซมเพียงไม่กี่พันครั้งจากกว่าหนึ่งล้านแบบ ในปัญหาจริงที่ต้นทุนมีผลของการเปิดวาล์วไอน้ำ (Valve-point Effect) หรือมีเขตห้ามเดินเครื่อง สูตรต้นทุนส่วนเพิ่มเท่ากันใช้ไม่ได้ตรง ๆ แต่ GA ยังใช้ได้เหมือนเดิม เพียงเปลี่ยนฟังก์ชันต้นทุน

**คำถามชวนคิด**

1. กราฟต้นทุนเฉลี่ย (สีส้ม) ไม่ลดลงเรียบเหมือนต้นทุนดีที่สุด เพราะอะไร
2. ถ้าตั้ง `PENALTY_PER_MW = 0` จะเกิดอะไรขึ้นกับคำตอบของ GA

## ส่วนที่ 7 แบบฝึกปฏิบัติ: ผลของอัตราการกลายพันธุ์

ใช้ GA หาค่าสูงสุดของฟังก์ชัน $f(x) = x\sin(10\pi x) + 1$ บนช่วง $[-1, 2]$ จากส่วนที่ 4 แล้วศึกษาผลของอัตราการกลายพันธุ์ $p_m$

**การเข้ารหัส** ใช้โครโมโซม 22 บิต แทนจำนวนเต็ม $n = 0, 1, \ldots, 2^{22} - 1$ แล้วแปลงเป็น

$$x = -1 + n \cdot \frac{3}{2^{22} - 1}$$

ความละเอียดจึงประมาณ $3 / 4.19 \times 10^6 \approx 7 \times 10^{-7}$ (เหมือนการเข้ารหัสในหนังสือของ Michalewicz, 1996)

### ข้อ 1 เขียนฟังก์ชันถอดรหัส

เขียน `decode_x(bits)` ให้คืนค่า $x$ ตามสูตรข้างบน แล้วเซลล์ถัดไปจะใช้ฟังก์ชันนี้สร้าง fitness และรัน GA

In [ ]:
N_BITS_X = 22


def decode_x(bits):
    """แปลงโครโมโซม 22 บิต (สตริง '0'/'1') เป็นค่า x ในช่วง [-1, 2]"""
    # TODO: แปลงบิตเป็นจำนวนเต็ม n ด้วย int(bits, 2) แล้วแปลงเป็น x ตามสูตร
    return None

In [ ]:
# เซลล์ตรวจคำตอบข้อ 1 (ไม่ต้องแก้)
def wavy_fitness(bits):
    return wavy(decode_x(bits))


if decode_x("0" * N_BITS_X) is None:
    print("ข้อ 1: ยังไม่ได้ทำ")
else:
    checks = [("0" * 22, -1.0), ("1" * 22, 2.0), ("1" + "0" * 21, -1 + 2 ** 21 * 3 / (2 ** 22 - 1))]
    ok = all(abs(decode_x(b) - x) < 1e-9 for b, x in checks)
    print("ข้อ 1:", "ผ่าน" if ok else "ยังไม่ผ่าน — ตรวจสูตรอีกครั้ง")
    best_x_bits, best_fit_x, _ = genetic_algorithm(wavy_fitness, N_BITS_X, pop_size=30, n_generations=50, pm=0.01)
    print(f"GA (pm = 0.01): x = {decode_x(best_x_bits):.4f}, f = {best_fit_x:.4f} "
          f"(ค่าสูงสุดวงกว้าง f = {F_BEST:.4f})")

### ข้อ 2 ทดลองเปลี่ยนอัตราการกลายพันธุ์

เขียนโค้ดรัน GA ด้วย $p_m$ ทุกค่าใน `PM_VALUES` ค่าละ 20 ครั้ง (ใช้ `seed` = 0 ถึง 19 เพื่อให้ผลซ้ำได้) พารามิเตอร์อื่นตามที่กำหนด แล้วเก็บผลลงลิสต์ `experiment_rows` หนึ่งแถวต่อหนึ่งค่า $p_m$ ประกอบด้วย

- `"pm"` — ค่า $p_m$
- `"f ที่ดีที่สุดเฉลี่ย"` — ค่าเฉลี่ยของ fitness ที่ดีที่สุดจาก 20 ครั้ง
- `"อัตราความสำเร็จ (%)"` — ร้อยละของครั้งที่ fitness ที่ดีที่สุด $\ge$ `GA_SUCCESS_LEVEL` (ต่างจากค่าสูงสุดวงกว้างไม่เกิน 0.0005 เข้มกว่าส่วนที่ 4 เพราะ GA ประเมินฟังก์ชันมากกว่า)
- `"f เฉลี่ยของประชากรรุ่นสุดท้าย"` — ค่าเฉลี่ยของ `history["mean"][-1]` จาก 20 ครั้ง

และเก็บประวัติ `history["best"]` ของทุกครั้งไว้ใน `convergence[pm]` (ลิสต์ของลิสต์) เพื่อใช้ในข้อ 3

In [ ]:
PM_VALUES = [0.0, 0.005, 0.02, 0.1, 0.3]
N_RUNS = 20
GA_SETTINGS = dict(n_bits=N_BITS_X, pop_size=30, n_generations=50, pc=0.9, elitism=1)
GA_SUCCESS_LEVEL = F_BEST - 0.0005

experiment_rows = []
convergence = {}

for pm in PM_VALUES:
    best_fits, final_means, histories = [], [], []
    for seed in range(N_RUNS):
        # TODO: เรียก genetic_algorithm(wavy_fitness, pm=pm, seed=seed, **GA_SETTINGS)
        #       แล้วเก็บ fitness ที่ดีที่สุด, history["mean"][-1] และ history["best"]
        pass
    # TODO: เพิ่มแถวผลลัพธ์ของ pm นี้ลงใน experiment_rows และเก็บ histories ลงใน convergence[pm]

experiment_table = pd.DataFrame(experiment_rows)
display(experiment_table)

### ข้อ 3 วาดกราฟการลู่เข้า

วาดกราฟเส้น 1 เส้นต่อ 1 ค่า $p_m$ แกนนอนคือรุ่น (0–50) แกนตั้งคือ **ค่าเฉลี่ยข้าม 20 ครั้ง** ของ fitness ที่ดีที่สุดในแต่ละรุ่น (ใช้ `np.mean(convergence[pm], axis=0)`) และลากเส้นประแสดง `F_BEST`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# TODO: วนทุก pm ใน PM_VALUES แล้ววาดค่าเฉลี่ยของ convergence[pm] ตามแกน axis=0
# TODO: วาดเส้นประแนวนอนที่ F_BEST, ตั้งชื่อแกน และแสดง legend
plt.show()

### ข้อ 4 วิเคราะห์ผล

ตอบคำถามต่อไปนี้จากตารางและกราฟที่ได้

1. เมื่อ $p_m = 0$ (ไม่มีการกลายพันธุ์เลย) GA มีพฤติกรรมอย่างไร เพราะอะไร
2. เมื่อ $p_m$ สูงมาก (0.3) ค่า "f เฉลี่ยของประชากรรุ่นสุดท้าย" ต่างจากค่า $p_m$ ต่ำอย่างไร GA ในกรณีนี้คล้ายวิธีค้นหาแบบใด
3. ในการทดลองนี้ $p_m$ ช่วงใดให้ผลดีที่สุด ถ้าความยาวโครโมโซมเพิ่มจาก 22 เป็น 44 บิต ค่าที่เหมาะสมควรเปลี่ยนไปอย่างไร (ใบ้: จำนวนบิตที่คาดว่าจะกลายพันธุ์ต่อโครโมโซมคือ $L \times p_m$)
4. GA ในการทดลองนี้ใช้ Elitism 1 ตัว ถ้าไม่ใช้ Elitism ผลของ $p_m = 0.3$ จะเป็นอย่างไร (ลองรันโดยตั้ง `elitism=0`)

*คำตอบ:* (เขียนคำตอบที่นี่)

## สรุป

- **Hill Climbing** เขียนง่าย ใช้หน่วยความจำน้อย แต่ผลขึ้นกับจุดเริ่มต้น ในตัวอย่างการจูน $K_p$ จุดเริ่ม 6 ใน 11 ค่าเท่านั้นที่ไปถึงยอดสูงสุด ($p = 6/11$) และ Random-restart ต้องทำเฉลี่ย $1/p \approx 1.8$ รอบ (ส่วนที่ 1)
- **Simulated Annealing** ยอมรับการย้ายที่แย่ลงด้วยความน่าจะเป็น $e^{\Delta E/T}$ จึงหนีจากค่าสูงสุดเฉพาะที่ได้ ตัวอย่างที่ 2.20 เป็นกรณีที่เลขสุ่มเป็นใจ การทดลองหลายครั้งแสดงว่าต้องลดอุณหภูมิช้าพอจึงจะสำเร็จสม่ำเสมอ (ส่วนที่ 2)
- **ปัญหา 8 ราชินี** Steepest-ascent สำเร็จราว 14% การยอมเดินด้านข้างเพิ่มเป็นราว 90% ขึ้นไป และ Random-restart สำเร็จเสมอแต่ใช้ก้าวมากกว่า ตรงกับตัวเลขที่หนังสืออ้างจาก Russell และ Norvig (ส่วนที่ 3)
- **ภูมิทัศน์หลายยอด** อย่าง $x\sin(10\pi x) + 1$ ทำให้ Hill Climbing ติดยอดเล็กเกือบทุกครั้ง ขณะที่ SA และ Random-restart ทำได้ดีกว่ามาก (ส่วนที่ 4)
- **Genetic Algorithm** ทำซ้ำตัวอย่างที่ 2.21 ได้ทุกตัวเลข และได้ $P_1 = 95$ MW ซึ่งใกล้คำตอบต้นทุนส่วนเพิ่มเท่ากัน $P_1 \approx 95.8$ MW ที่สุดในการเข้ารหัสนี้ เมื่อขยายเป็น 3 เครื่อง GA ได้ต้นทุนใกล้คำตอบของ Lambda Iteration มาก โดยไม่ต้องใช้อนุพันธ์ (ส่วนที่ 5–6)
- **อัตราการกลายพันธุ์** ต่ำเกินไปทำให้ลู่เข้าก่อนกำหนด สูงเกินไปทำให้กลายเป็นการค้นหาแบบสุ่ม ค่าที่เหมาะสมมักอยู่ราว $1/L$ (แบบฝึกปฏิบัติ)

**ก้าวต่อไป:** ปฏิบัติการที่ 6 จะศึกษาการค้นหาเมื่อมีคู่แข่งขัน (Minimax และ Alpha-Beta) และปัญหาที่มีข้อจำกัด (CSP) ตามหัวข้อ 2.5–2.6

## เอกสารอ้างอิง

- Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. บทที่ 4.1 (Local Search, 8-Queens)
- Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983). Optimization by simulated annealing. *Science, 220*(4598), 671–680.
- Michalewicz, Z. (1996). *Genetic Algorithms + Data Structures = Evolution Programs* (3rd ed.). Springer. (ฟังก์ชันทดสอบ $x\sin(10\pi x) + 1$)
- Wood, A. J., Wollenberg, B. F., & Sheblé, G. B. (2014). *Power Generation, Operation, and Control* (3rd ed.). Wiley. บทที่ 3 (Economic Dispatch, ตัวอย่าง 3A)